### eBird Data Transformation: Great-tailed Grackle (*Quiscalus mexicanus*)
---
**Capstone Project · DataTalks.Club 2026**  
**Author:** Victoria Torreno  
**Source:** [eBird API 2.0](https://documenter.getpostman.com/view/664302/S1ENwy59)  
**Goal:** Prototype Bronze → Silver transformations including type casting, schema standardization, column renaming, and deduplication before implementing in the production Spark job.

#### Setup Spark Session and Load Raw GCS Data

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import count, when, col
from dotenv import load_dotenv
from pathlib import Path 

In [ ]:
load_dotenv()

In [ ]:
GCS_BUCKET = os.getenv('GCS_BUCKET')
BRONZE_PATH = f'gs://{GCS_BUCKET}/bronze_ebird/ebird_observations'
JAR_PATH = str(Path().resolve().parents[1] / 'jars' / 'gcs-connector.jar')

In [ ]:
# initialize Spark session with GCS connector
spark = SparkSession.builder \
    .appName('ebird_bronze_to_silver') \
    .config('spark.jars', JAR_PATH) \
    .config('spark.hadoop.fs.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem') \
    .config('spark.hadoop.fs.AbstractFileSystem.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS') \
    .config('spark.hadoop.google.cloud.auth.service.account.enable', 'true') \
    .config('spark.hadoop.google.cloud.auth.service.account.json.keyfile', os.getenv('GOOGLE_APPLICATION_CREDENTIALS')) \
    .config('spark.ui.showConsoleProgress', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark initialized with local GCS connector.")

In [ ]:
# read raw parquet from GCS bronze
bronze_df = spark.read.parquet(BRONZE_PATH)
print(f'Rows: {bronze_df.count():,}')
print(f'Columns: {len(bronze_df.columns)}')

#### Schema Enforcement & Column Pruning
Selecting core fields from the raw eBird schema. Drops dlt internals and fields not relevant to spatial, temporal, or behavioral analysis.

In [ ]:
# preview raw schema
bronze_df.printSchema()
bronze_df.limit(5).toPandas()

In [ ]:
SILVER_COLUMNS = [
    # identifiers
    'sub_id',
    'loc_id',

    # location
    'loc_name',
    'lat',
    'lng',

    # temporal
    'obs_dt',

    # behavioral
    'how_many',
    'obs_valid',
    'obs_reviewed',
    'location_private',

    # taxonomy
    'species_code',
    'com_name',
    'sci_name',

    # provenance
    'region_code',
]

In [ ]:
silver_df = bronze_df.select(SILVER_COLUMNS)
print(f'Columns: {len(silver_df.columns)}')
silver_df.printSchema()

#### Deduplication
Ensures each checklist sighting is counted once across all region fetches.

In [ ]:
initial_count = silver_df.count()
silver_df = silver_df.dropDuplicates(['sub_id'])
current_count = silver_df.count()
print(f'Removed {initial_count - current_count:,} duplicate records.')
print(f'Remaining records: {current_count:,}')

#### Filter Invalid Records
Dropping records with missing coordinates. Required for all spatial analysis downstream.

In [ ]:
initial_count = silver_df.count()
silver_df = silver_df.dropna(subset=['lat', 'lng', 'obs_dt'])
current_count = silver_df.count()
print(f"Dropped {initial_count - current_count:,} records with missing coordinates or date.")
print(f"Remaining records: {current_count:,}")

#### Type Casting & Column Renaming
Casting `obs_dt` to date, `how_many` to integer, extracting `year` and `month` for partitioning, and renaming columns for clarity and consistency with the Silver layer naming conventions.

In [ ]:
# type casting
silver_df = silver_df \
    .withColumn('obs_dt', f.col('obs_dt').cast('date')) \
    .withColumn('how_many', f.col('how_many').cast('integer'))

In [ ]:
# extract year and month for partitioning
silver_df = silver_df \
    .withColumn('year', f.year(f.col('obs_dt'))) \
    .withColumn('month', f.month(f.col('obs_dt')))

In [ ]:
# column renaming aligned with GBIF Silver schema for cross-source joins
silver_df = silver_df \
    .withColumnRenamed('sub_id', 'checklist_id') \
    .withColumnRenamed('loc_id', 'location_id') \
    .withColumnRenamed('loc_name', 'location_name') \
    .withColumnRenamed('lat', 'latitude') \
    .withColumnRenamed('lng', 'longitude') \
    .withColumnRenamed('obs_dt', 'event_date') \
    .withColumnRenamed('how_many', 'count') \
    .withColumnRenamed('obs_valid', 'is_valid') \
    .withColumnRenamed('obs_reviewed', 'is_reviewed') \
    .withColumnRenamed('com_name', 'common_name') \
    .withColumnRenamed('sci_name', 'scientific_name') \
    .withColumnRenamed('region_code', 'state')

silver_df.printSchema()

#### Null Handling
Auditing null counts to inform handling decisions, then applying fixes before writing to Silver.

In [ ]:
# null audit before handling
null_counts = silver_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in silver_df.columns
])
null_counts.toPandas().T.rename(columns={0: 'null_count'})

In [ ]:
# fill null count with 1 (minimum presence assumption)
silver_df = silver_df.fillna({'count': 1})

> **Note:** `count` nulls represent presence-only observations where flock size was not recorded. Filling with 1 represents minimum presence. Unlike GBIF, eBird `count` is only 3.8% null, making it reliable for flock size analysis.

#### Null Audit
Verifying null counts after all transformations before writing to Silver.

In [ ]:
null_counts_final = silver_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in silver_df.columns
])
null_counts_final.toPandas().T.rename(columns={0: 'null_count'})

#### Final Validation
A last look at the data before committing to GCS Silver.

In [ ]:
# validate schema
silver_df.printSchema()

In [ ]:
df = silver_df.toPandas()
df.head()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# sightings per state
silver_df.groupBy('state').count().orderBy('count', ascending=False).toPandas()

In [ ]:
# top 10 most common flock sizes
silver_df.groupBy('count') \
    .agg(f.count('*').alias('sightings')) \
    .orderBy('count') \
    .limit(10) \
    .toPandas()

In [ ]:
# sightings by year and month
silver_df.groupBy('year', 'month').count().orderBy('year', 'month').toPandas()

#### Write to GCS Silver
Writing the cleaned and transformed dataset to GCS Silver as Parquet, partitioned by year and month for efficient downstream querying.

In [ ]:
SILVER_PATH = f'gs://{GCS_BUCKET}/silver_ebird'

silver_df.write \
    .mode('overwrite') \
    .partitionBy('year', 'month') \
    .parquet(SILVER_PATH)

print(f"Silver data written to: {SILVER_PATH}")
print(f"Final row count: {silver_df.count():,}")